In [14]:
import tensorflow as tf 
from collections import Counter
from tensorflow.keras import models, layers 

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "../dataset/train",
    image_size=(224, 224),
    batch_size=32
)

for images, labels in train_ds.take(1):
    print(images.shape, labels.shape)

Found 8582 files belonging to 2 classes.


2025-11-13 06:34:19.113414: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-13 06:34:19.391357: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [8582]
	 [[{{node Placeholder/_0}}]]
2025-11-13 06:34:19.397621: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [8582]
	 [[{{node Placeholder/_0}}]]


(32, 224, 224, 3) (32,)


In [5]:
class_names = train_ds.class_names
print(f"The class name is {class_names}")

The class name is ['negative', 'positive']


In [6]:
len(class_names)

2

In [7]:
label_counter = Counter()
for _,labels in train_ds:
    label_counter.update(labels.numpy())

print(label_counter)

2025-11-13 06:34:23.125169: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [8582]
	 [[{{node Placeholder/_4}}]]
2025-11-13 06:34:23.132099: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [8582]
	 [[{{node Placeholder/_4}}]]


Counter({0: 4291, 1: 4291})


In [8]:
normalization_layer = tf.keras.layers.Rescaling(1./255)


In [9]:
train_ds = train_ds.map(lambda x,y :(normalization_layer(x),(y)))

In [13]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)

In [15]:
model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    
    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    
    # Block 4
    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.3),
    
    # Flatten and Dense layers
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])


2025-11-13 06:47:01.331877: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 51380224 exceeds 10% of free system memory.
2025-11-13 06:47:01.497370: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 51380224 exceeds 10% of free system memory.
2025-11-13 06:47:01.552945: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 51380224 exceeds 10% of free system memory.


In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)


In [19]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    "../dataset/train",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)
val_ds = val_ds.map(lambda x,y:(normalization_layer(x), y)).prefetch(AUTOTUNE)


Found 8582 files belonging to 2 classes.
Using 1716 files for validation.
